# Single Head GAT using NUMPY
Used Only Numpy , fully custom Single Head GAT .

## Complete Mathametical Explanation :

For $node (i)$ :

$x_i \xrightarrow{W} h_i \quad (h_i = Wx_i)$

For every $neighbor (j)$ :

$[h_i || h_j] \xrightarrow{a^T} e_{ij} \quad (e_{ij} = \text{LeakyReLU}(a^T[h_i || h_j]))$

Normalize :

$e_{ij} \xrightarrow{Softmax} \alpha_{ij} \quad (\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}_i} \exp(e_{ik})})$

Aggregate :

$h_i' = \sum_{j \in \mathcal{N}_i} \alpha_{ij} h_j$

In [ ]:
def softmax(x):
  return np.exp(x)/np.sum(np.exp(x) , axis = -1 , keepdims = True)

In [ ]:
alpha = 0.2
def Leaky_RELU(x):
  return np.where(x>0 , x , alpha*x)

## Simple Workflow

1.  **Linear Transformation**: Transform node features using a weight matrix `W`.
2.  **Attention Coefficients**: Calculate attention coefficients `e_ij` between a node `i` and its neighbors `j` using a shared attention mechanism `a^T`.
3.  **Normalization**: Normalize the attention coefficients using a Softmax function to get `α_ij`.
4.  **Aggregation**: Aggregate the neighbor features, weighted by the normalized attention coefficients, to produce the new node features `h'_i`.

In [ ]:
import torch
import numpy as np

def custom_GAT():
   # --- Graph setup ---
    # Adjacency list (including self-loops)
    #   0 -- 1 -- 2
    #        |
    #        3
    #including the self loop
  adj = {
      0: [0, 1],
      1: [0, 1,2,3],
      2: [1, 2],
      3: [1, 3]
  }
  node = 4
  F_in = 5
  F_out = 2
  X = np.random.randn(node , F_in).astype(np.float32)
  W = np.random.randn(F_in,F_out).astype(np.float32) * 0.1
  A = np.random.randn(2*F_out).astype(np.float32) * 0.1

  H = X @ W # node * F_out
  output = np.zeros_like(H) # Initialize output array
  for i in range(node):
    neighbour = adj[i]
    h_i = H[i]
    h_j = H[neighbour]
    # Tile h_i to match the number of neighbors for concatenation
    h_i_tiled = np.tile(h_i, (len(neighbour), 1))
    concat = np.concatenate([h_i_tiled, h_j], axis  = 1)
    e= Leaky_RELU(concat @ A)
    atten = softmax(e)
    output[i] = atten @ h_j

  # output = Leaky_RELU(output)
  print(f"Input  X shape : {X.shape}")
  print(f"Output H shape : {output.shape}")
  print(f"Output (first 2 nodes):\n{output[:2]}")
  return output


In [ ]:
custom_GAT()

Input  X shape : (4, 5)
Output H shape : (4, 2)
Output (first 2 nodes):
[[ 0.1422984   0.1428289 ]
 [-0.09828871  0.06970196]]


array([[ 0.1422984 ,  0.1428289 ],
       [-0.09828871,  0.06970196],
       [ 0.03150717,  0.1960493 ],
       [-0.10006706,  0.0620936 ]], dtype=float32)

In [ ]:
print(np.array([1,2,3]).shape)

(3,)


In [ ]:
print(np.array([[1],[2],[3]]).shape)

(3, 1)


In [ ]:
import numpy as np
x = np.array([1,2,3]) # 1*3 matrix
y  = np.array([[1,2,3],[4,5,6],[7,8,9]]) # 3*3 matrix
print(x.shape)
print(y.shape)
mul = y @ x
print(mul)
print(mul.shape)

(3,)
(3, 3)
[14 32 50]
(3,)


In [ ]:
concat = [[1,2,3,4],[5,6,7,8],[9,1,2,3],[4,5,6,7]] # 4*4
A = [10,20,30,40] # (1*4)
concat = np.array(concat)
A = np.array(A)
print(concat.shape)
print(A.shape)
mul = concat @ A
print(mul)
print(mul.shape)

(4, 4)
(4,)
[300 700 290 600]
(4,)
